# Waste Classification Training Notebook

This notebook trains an image classifier using the dataset available in dataset/TrashType_Image_Dataset and saves the trained model in artifacts/waste_classifier.keras.

Expected dataset structure:
- dataset/TrashType_Image_Dataset/cardboard
- dataset/TrashType_Image_Dataset/glass
- dataset/TrashType_Image_Dataset/metal
- dataset/TrashType_Image_Dataset/paper
- dataset/TrashType_Image_Dataset/plastic
- dataset/TrashType_Image_Dataset/trash

In [ ]:
from pathlib import Path
import json
import numpy as np
import tensorflow as tf

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Paths
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'dataset' / 'TrashType_Image_Dataset'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_PATH = ARTIFACT_DIR / 'waste_classifier.keras'
LABELS_PATH = ARTIFACT_DIR / 'waste_classifier_labels.json'

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Dataset path:', DATA_DIR)
print('Model output:', MODEL_PATH)

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Dataset folder not found: {DATA_DIR}')

class_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
if not class_dirs:
    raise ValueError(f'No class folders found in: {DATA_DIR}')

print('Detected classes and image counts:')
for c in class_dirs:
    count = len(list(c.glob('*.jpg'))) + len(list(c.glob('*.jpeg'))) + len(list(c.glob('*.png')))
    print(f'- {c.name}: {count}')

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.2

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Classes:', class_names)
print('Number of classes:', num_classes)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
EPOCHS_HEAD = 8

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)

In [ ]:
# Fine-tuning last layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_FINE = 5
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=0)
print(f'Validation accuracy: {acc:.4f}')

model.save(MODEL_PATH)
print('Saved model to:', MODEL_PATH)

label_info = {
    'class_names': class_names,
    'image_size': list(IMG_SIZE),
    'num_classes': num_classes
}
LABELS_PATH.write_text(json.dumps(label_info, indent=2), encoding='utf-8')
print('Saved labels to:', LABELS_PATH)

In [ ]:
def classify_image(image_path: str):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    arr = tf.expand_dims(arr, axis=0)
    probs = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))
    return {
        'predicted_class': class_names[idx],
        'confidence': float(probs[idx]),
        'all_scores': {name: float(score) for name, score in zip(class_names, probs)}
    }

sample_image = next((DATA_DIR / class_names[0]).glob('*.jpg'), None)
if sample_image:
    print('Sample image:', sample_image)
    print(classify_image(str(sample_image)))
else:
    print('No sample image found for quick prediction test.')

## Integration Note

This model is trained on 6 classes from TrashType dataset.
If your API expects only 3 classes (Organic, Recyclable, Hazardous), then either:
1. Retrain with 3 folders matching those classes, or
2. Add a mapping layer in API from 6-class outputs to your 3 target categories.